# Land-Variable Lead-Time ACC Skill Maps

This is the land companion to `1a_refactor_atm_leadtime_acc_skill_map.ipynb`. It
prepares provenance-validated seasonal inputs, enforces common lead-specific target-
year cohorts across E3SM cases, computes cached ACC metrics, and writes comparison
figures. Preparation is planned independently for the reference and every case/month,
so restarts open only raw inputs whose prepared caches need rebuilding.

Prepared inputs and skills use the same source-first hierarchy as `1a`:
`<S2D_DIAG_ROOT>/<source-or-case>/leadtime_acc/{inputs,skill}/land/<field>/`.


In [5]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display

from esp_lab import land_prepared_skill, land_skill, stats
from esp_lab.leadtime_plot_utils import (
    add_lead_badge,
    add_missing_map_panel,
    seasonal_label,
    style_global_map_axis,
)
from esp_lab.paths import leadtime_acc_dir
from esp_lab.utils.filename_utils import figure_filename, safe_token
from esp_lab.utils.netcdf_utils import atomic_to_netcdf, load_netcdf
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## User setup

Choose the land field, archives, evaluation period, cache policy, Dask resources, and
outputs here. `prepared_land.mode="auto"` with inventory identity is the normal
production setting. After an inventory run, `source_identity_mode="snapshot"` with
`prepared_land.mode="require"` provides an archive-free restart from the exact prepared
inputs. `smoke_mode=True` selects a small but archive-backed two-case run.


In [6]:
# Select H2OSNO, H2OSOI, or a fully configured TWS reference.
field = "H2OSNO"
SOIL_DEPTH_RANGE_M = (0.0, 1.6) if field == "H2OSOI" else None
TARGET_DLAT = 1.0
TARGET_DLON = 1.0
GRID_TAG = land_prepared_skill.target_grid_tag(TARGET_DLAT, TARGET_DLON)
REGRIDDING_METHOD = "conservative"

RUN = {
    "smoke_mode": False,
    "initialization_years": (1980, 2011), #(1980, 2018),
    "init_months": [5, 11],
    "climatology_years": (1981, 2010),
    "ensemble_members": [f"EN{i:02d}" for i in range(10)],
    "monthly_nlead": 24,
    "seasonal_nlead": 8,
    "detrend": True,
    "strict_member_completeness": True,
    "force_compute": False,
    "raw_model_chunks": {"Y": 3, "L": 24, "M": 2, "lat": 90, "lon": 180},
    "model_chunks": {"Y": -1, "L": 4, "M": 2, "lat": 45, "lon": 90},
    "reference_chunks": {"time": -1, "lat": 45, "lon": 90},
}

E3SM_CASES = {
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
        "source_revision": "post_process_v1",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
        "source_revision": "post_process_v1",
    },
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "cache_tag": "4DEnVarOcn",
        "display_name": "E3SMv3-4DEnVarOcn",
        "source_revision": "post_process_v1",
    },
}

REFERENCE_CONFIGS = {
    "H2OSNO": {
        "path": "/global/cfs/cdirs/e3sm/zhan391/data/C3S_SWE/1x1/monthly/swe_*.nc",
        "variable": "swe", "product": "C3S_SWE",
        "documentation": "Copernicus Climate Change Service snow water equivalent",
        "source_revision": "c3s_swe_archive_2026-08-26",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": False, "already_seasonal": False,
        "mask_negative_categorical_flags": True,
        "require_complete_calendar_months": True,
        "retain_missing_seasons": True,
        "restrict_model_to_reference_months": True,
    },
    "TWS": {
        "path": "", "variable": "tws", "product": "CONFIGURE_ME",
        "documentation": "", "source_revision": "configure_me",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": True, "already_seasonal": False,
        "restrict_model_to_reference_months": False,
    },
    "H2OSOI": {
        "path": "/global/cfs/cdirs/e3sm/zhan391/data/CPC_SOM/monthly/soilw_*.nc",
        "variable": "soilw", "product": "CPC_Soil_Moisture_V2",
        "documentation": "https://www.cpc.ncep.noaa.gov/soilmst/descrip.htm",
        "source_revision": "cpc_soil_moisture_v2_archive_2026-08-26",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": False, "already_seasonal": False,
        "vertical_dim": None, "layer_bounds_variable": None,
        "represented_depth_range_m": (0.0, 1.6),
        "restrict_model_to_reference_months": False,
    },
}
reference_cfg = REFERENCE_CONFIGS[field]
REFERENCE_PRODUCT = reference_cfg["product"]
if REFERENCE_PRODUCT == "CONFIGURE_ME":
    raise ValueError("Configure the TWS reference product and raw-data contract")

DASK_SETTINGS = {
    "enabled": True, "cluster_type": "local", "workers": 4,
    "memory_limit": "4GB",
}

CACHE_SETTINGS = {
    "prepared_land": {"mode": "auto"},  # auto, rebuild, or require
    "source_identity_mode": "inventory",  # inventory, snapshot, or revision
    "cleanup_temp_files": True,
    "temp_file_max_age_hours": 24.0,
}
PATHS = {
    "raw_model_root": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
    "staged_input_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
    "cache_output_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
    "figure_outdir": "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag",
}
if RUN["smoke_mode"]:
    RUN.update(
        initialization_years=(2000, 2004), init_months=[11],
        climatology_years=(2000, 2004),
        ensemble_members=["EN00", "EN01"], monthly_nlead=6, seasonal_nlead=2,
    )
    print("ESP-Lab land smoke mode: two cases, one month, five years, two members/leads.")

RAW_MODEL_ROOT = Path(PATHS["raw_model_root"])
STAGED_INPUT_ROOT = Path(PATHS["staged_input_root"])
S2D_DIAG_ROOT = Path(PATHS["cache_output_root"])
FIGURE_OUTDIR = Path(PATHS["figure_outdir"])
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

EVALUATION_PROTOCOL = (
    f"land_acc_init{RUN['initialization_years'][0]}-{RUN['initialization_years'][1]}_"
    f"clim{RUN['climatology_years'][0]}-{RUN['climatology_years'][1]}_"
    f"per-lead_{'detrend' if RUN['detrend'] else 'nodetrend'}"
)

REFERENCE_DISPLAY_NAMES = {
    "CPC_Soil_Moisture_V2": "CPC V2",
    "C3S_SWE": "C3S SWE",
}
REFERENCE_DISPLAY_NAME = REFERENCE_DISPLAY_NAMES.get(
    REFERENCE_PRODUCT, REFERENCE_PRODUCT.replace("_", " "),
)
if field == "H2OSOI":
    if REFERENCE_PRODUCT == "CPC_Soil_Moisture_V2":
        REFERENCE_LEVEL_DEPTH = "0–1.6m"
        REFERENCE_CAPTION_NAME = "0–1.6m soil moisture"
    elif REFERENCE_PRODUCT == "C3S_SWE":
        REFERENCE_LEVEL_DEPTH = "0-5cm"
        REFERENCE_CAPTION_NAME = "upper ~5 cm"

init_month_names = {
    1: "JAN", 2: "FEB", 3: "MAR", 4: "APR", 5: "MAY", 6: "JUN",
    7: "JUL", 8: "AUG", 9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC",
}
MAP_STYLE = {
    "lon_ticks": [-160, -80, 0, 80, 160],
    "lat_ticks": [-60, -30, 0, 30, 60],
    "tick_length": 2.5, "tick_width": 0.5,
    "grid_linewidth": 0.35, "grid_color": "0.35",
    "grid_alpha": 0.35,
} 

FIGURE_SETTINGS = {
    "acc": {
        "significance_mask": True, "significance_level": 0.1,
        "mask_negative_acc": False, "cmap": "blue2red_acc",
        "color_interval": 0.1, "color_min": -1.0, "color_max": 1.0,
        "color_cutoff": 0.5, "font_size": 16,
        "column_width": 4.4, "row_height": 2.8,
        "section_gap": 0.035, "dpi": 300,
    },
    "difference": {
        "color_interval": 0.05, "color_min": -0.5, "color_max": 0.5,
        "color_cutoff": 0.25, "cmap": "blue2red",
        "column_width": 4.0, "row_height": 1.5, "font_size": 12,
        "title_template": "ACC Differences ({variable}, ref: {reference})",
        "subtitle_template": "{left} − {right}",
        "colorbar_label": r"${\Delta}$ACC",
        "title_font_scale": 1.0, "subtitle_font_scale": 0.85,
        "colorbar_tick_font_scale": 0.8,
        "layout_edges": (0.035, 0.065, 0.995),
        "header_top_pad_points": 3.0, "title_subtitle_gap_points": 3.0,
        "header_panel_gap_points": 8.0, "layout_pad": 0.3,
        "layout_h_pad": 0.15, "layout_w_pad": 0.1,
        "subplot_bottom": 0.075, "subplot_hspace": 0.015,
        "subplot_wspace": 0.015,
        "colorbar_rect": (0.28, 0.018, 0.44, 0.012), "dpi": 300,
    },
}


## Managed Dask resources

A local production run uses four one-thread workers with the shared-node safety cap.
Rerunning this cell closes any previous notebook cluster and tracked datasets first.


In [7]:
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(DaskConfig(
        cluster_type=DASK_SETTINGS["cluster_type"],
        workers=DASK_SETTINGS["workers"],
        memory_limit=DASK_SETTINGS["memory_limit"],
    )) if DASK_SETTINGS["enabled"] else (None, None),
)
if client is not None:
    display(client)



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
This can easily exceed memory/CPU limits and crash your Jupyter kernel.
We highly recommend launching Jupyter on an Exclusive Perlmutter Compute Node.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Scheduler: tcp://127.0.0.1:40513
Connected workers: 4
Dashboard: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40513,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42499,Total threads: 1
Dashboard: http://127.0.0.1:41043/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:44537,


## Prepare or reuse analysis-ready inputs

Planning is independent for the reference and every model case/month. Inventory mode
records exact file identities and snapshots before any expensive preparation. Snapshot
mode never resolves raw paths and requires compatible prepared caches. Dataset contracts
validate dimensions, members, years, grid, units, depth treatment, and provenance.


In [ ]:
prepared_mode = CACHE_SETTINGS["prepared_land"]["mode"]
if prepared_mode not in {"auto", "rebuild", "require"}:
    raise ValueError("prepared_land.mode must be 'auto', 'rebuild', or 'require'")
source_identity_mode = CACHE_SETTINGS["source_identity_mode"]
if source_identity_mode not in {"inventory", "snapshot", "revision"}:
    raise ValueError("source_identity_mode must be inventory, snapshot, or revision")
if source_identity_mode == "snapshot" and prepared_mode != "require":
    raise ValueError("Snapshot mode requires prepared_land.mode='require'")

if "workflow_resources" not in globals():
    raise RuntimeError("Run the managed Dask setup cell before preparation")
netcdf_write_options = {
    "cleanup_temporary": CACHE_SETTINGS["cleanup_temp_files"],
    "temp_file_max_age_hours": CACHE_SETTINGS["temp_file_max_age_hours"],
}
years = np.arange(RUN["initialization_years"][0], RUN["initialization_years"][1] + 1)
target_grid = land_prepared_skill.target_grid(TARGET_DLAT, TARGET_DLON)
SOURCE_SNAPSHOT_DIR = S2D_DIAG_ROOT / "source_inventory_snapshots" / "land"
DEPTH_TOKEN = land_skill.land_depth_token(field, SOIL_DEPTH_RANGE_M)
reference_path = land_skill.staged_land_input_path(
    STAGED_INPUT_ROOT, REFERENCE_PRODUCT, field, GRID_TAG,
    depth_range_m=SOIL_DEPTH_RANGE_M,
)
model_input_paths = {
    case_name: {
        init_month: land_skill.staged_land_input_path(
            STAGED_INPUT_ROOT, case_cfg["cache_tag"], field, GRID_TAG,
            init_month=init_month, depth_range_m=SOIL_DEPTH_RANGE_M,
        )
        for init_month in RUN["init_months"]
    }
    for case_name, case_cfg in E3SM_CASES.items()
}

reference_files = (
    land_prepared_skill.resolve_reference_files(reference_cfg["path"])
    if source_identity_mode == "inventory" else []
)
reference_raw_identity = land_prepared_skill.raw_source_identity(
    paths=reference_files,
    logical_identity={
        "product": REFERENCE_PRODUCT, "variable": reference_cfg["variable"],
        "field": field, "path_pattern": reference_cfg["path"],
    },
    source_revision=reference_cfg["source_revision"],
    mode=source_identity_mode,
    inventory_root=Path(reference_cfg["path"]).parent,
    snapshot_dir=SOURCE_SNAPSHOT_DIR,
)
reference_month_policy = (
    "complete centered 3-month windows; retain explicit missing seasons"
    if reference_cfg.get("require_complete_calendar_months", False)
    else "all seasonal center months"
)
reference_expected_attrs = land_prepared_skill.expected_attrs(
    field=field, source_kind="reference", source_name=REFERENCE_PRODUCT,
    source_data_identity=reference_raw_identity,
    initialization_years=RUN["initialization_years"],
    climatology_years=RUN["climatology_years"],
    ensemble_members=RUN["ensemble_members"], monthly_nlead=RUN["monthly_nlead"],
    target_grid_name=GRID_TAG, regridding_method=REGRIDDING_METHOD,
    output_units=reference_cfg["output_units"],
    reference_is_anomaly=reference_cfg["is_anomaly"],
    depth_range_m=SOIL_DEPTH_RANGE_M,
    reference_month_policy=reference_month_policy,
)
reference_ok, reference_reason = land_prepared_skill.prepared_cache_status(
    reference_path, reference_expected_attrs, grid=target_grid
)
if prepared_mode == "require" and not reference_ok:
    raise FileNotFoundError(f"Required prepared reference is unavailable: {reference_reason}")
if prepared_mode == "rebuild" or not reference_ok:
    if not reference_files:
        reference_files = land_prepared_skill.resolve_reference_files(reference_cfg["path"])
    print(f"Preparing reference ({reference_reason}): {reference_path}")
    land_prepared_skill.prepare_reference_cache(
        paths=reference_files, cfg=reference_cfg, field=field,
        depth_range_m=SOIL_DEPTH_RANGE_M, grid=target_grid,
        expected=reference_expected_attrs, output_path=reference_path,
        write_options=netcdf_write_options,
    )
else:
    print("Reusing prepared reference:", reference_path)

reference_dataset = xr.open_dataset(reference_path, chunks=RUN["reference_chunks"])
workflow_resources.track(reference_dataset)
land_prepared_skill.validate_dataset(
    reference_dataset, reference_expected_attrs, grid=target_grid
)
reference = reference_dataset[field]
reference_is_anomaly = reference_cfg["is_anomaly"]
reference_data_identity = land_prepared_skill.prepared_data_identity(
    reference_expected_attrs
)

model_expected_attrs = {}
model_data_identity_by_case_month = {}
for case_name, case_cfg in E3SM_CASES.items():
    model_expected_attrs[case_name] = {}
    model_data_identity_by_case_month[case_name] = {}
    for init_month in RUN["init_months"]:
        model_files = (
            land_prepared_skill.resolve_model_files(
                data_dir=RAW_MODEL_ROOT, case_prefix=case_cfg["case_prefix"],
                members=RUN["ensemble_members"], years=years,
                init_month=init_month, field=field, nlead=RUN["monthly_nlead"],
            )
            if source_identity_mode == "inventory" else []
        )
        raw_identity = land_prepared_skill.raw_source_identity(
            paths=model_files,
            logical_identity={
                "case": case_name, "case_prefix": case_cfg["case_prefix"],
                "field": field, "init_month": init_month,
                "years": list(map(int, years)),
                "members": list(RUN["ensemble_members"]),
                "monthly_nlead": int(RUN["monthly_nlead"]),
            },
            source_revision=case_cfg["source_revision"],
            mode=source_identity_mode, inventory_root=RAW_MODEL_ROOT,
            snapshot_dir=SOURCE_SNAPSHOT_DIR,
        )
        expected = land_prepared_skill.expected_attrs(
            field=field, source_kind="model", source_name=case_name,
            source_data_identity=raw_identity, case_prefix=case_cfg["case_prefix"],
            init_month=init_month, initialization_years=RUN["initialization_years"],
            climatology_years=RUN["climatology_years"],
            ensemble_members=RUN["ensemble_members"], monthly_nlead=RUN["monthly_nlead"],
            target_grid_name=GRID_TAG, regridding_method=REGRIDDING_METHOD,
            output_units=reference_cfg["output_units"],
            reference_is_anomaly=reference_cfg["is_anomaly"],
            depth_range_m=SOIL_DEPTH_RANGE_M,
            reference_month_policy=reference_month_policy,
        )
        path = model_input_paths[case_name][init_month]
        compatible, reason = land_prepared_skill.prepared_cache_status(
            path, expected, grid=target_grid
        )
        if prepared_mode == "require" and not compatible:
            raise FileNotFoundError(f"Required prepared model input {path} is unavailable: {reason}")
        if prepared_mode == "rebuild" or not compatible:
            print(f"Preparing {case_name}, init={init_month} ({reason}): {path}")
            land_prepared_skill.prepare_model_cache(
                data_dir=RAW_MODEL_ROOT, case_prefix=case_cfg["case_prefix"],
                members=RUN["ensemble_members"], years=years, init_month=init_month,
                field=field, monthly_nlead=RUN["monthly_nlead"],
                monthly_chunks=RUN["raw_model_chunks"],
                depth_range_m=SOIL_DEPTH_RANGE_M, reference=reference,
                restrict_to_reference_months=reference_cfg.get(
                    "restrict_model_to_reference_months", False
                ),
                climatology_years=RUN["climatology_years"],
                strict_member_completeness=RUN["strict_member_completeness"],
                grid=target_grid, expected=expected, output_path=path,
                write_options=netcdf_write_options,
            )
        else:
            print(f"Reusing prepared model input: {case_name}, init={init_month}: {path}")
        model_expected_attrs[case_name][init_month] = expected
        model_data_identity_by_case_month[case_name][init_month] = (
            land_prepared_skill.prepared_data_identity(expected)
        )

print(f"Prepared-input mode: {prepared_mode}; source identity: {source_identity_mode}")


Reusing prepared reference: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/C3S_SWE/leadtime_acc/inputs/land/H2OSNO/C3S_SWE_H2OSNO_seasonal_1x1deg_cell_centered.nc


In [ ]:
forecast_by_case_month = {}
valid_time_by_case_month = {}
expected_years_by_case_month = {}
for case_name, case_cfg in E3SM_CASES.items():
    forecast_by_case_month[case_name] = {}
    valid_time_by_case_month[case_name] = {}
    expected_years_by_case_month[case_name] = {}
    for init_month in RUN["init_months"]:
        path = model_input_paths[case_name][init_month]
        ds = xr.open_dataset(path, chunks=RUN["model_chunks"])
        workflow_resources.track(ds)
        land_prepared_skill.validate_dataset(
            ds, model_expected_attrs[case_name][init_month], grid=target_grid
        )
        forecast = ds[field]
        land_skill.validate_land_reference_compatibility(forecast, reference)
        forecast, valid_time, dropped_leads = land_skill.retain_valid_seasonal_leads(
            forecast, ds.time
        )
        valid_time = valid_time.load()
        expected_years = land_skill.validate_hindcast_evaluation_setup(
            forecast, valid_time, init_month=init_month,
            initialization_years=RUN["initialization_years"],
            expected_members=RUN["ensemble_members"],
            climatology_years=RUN["climatology_years"],
            require_complete_member_grid=RUN["strict_member_completeness"],
        )
        forecast_by_case_month[case_name][init_month] = forecast
        valid_time_by_case_month[case_name][init_month] = valid_time
        expected_years_by_case_month[case_name][init_month] = expected_years
        print("Validated:", case_name, init_month, path, "dropped leads:", dropped_leads)

VALID_LEADS_BY_MONTH = {}
for init_month in RUN["init_months"]:
    lead_coordinates = {
        tuple(forecast_by_case_month[case][init_month].L.values.tolist())
        for case in E3SM_CASES
    }
    if len(lead_coordinates) != 1:
        raise ValueError(
            f"Models do not share valid leads for init={init_month}: {lead_coordinates}"
        )
    VALID_LEADS_BY_MONTH[init_month] = list(next(iter(lead_coordinates)))
print("Valid seasonal leads by initialization month:", VALID_LEADS_BY_MONTH)


## Establish common target-year cohorts

For each initialization month and lead, all E3SM cases use the same intersection of model and reference target years. This prevents apparent skill differences caused only by unequal temporal samples.

In [ ]:
common_years_by_month = {}
model_climatology_count_by_month = {}
for init_month in RUN["init_months"]:
    model_fields = {
        case_name: forecast_by_case_month[case_name][init_month]
        for case_name in E3SM_CASES
    }
    model_times = {
        case_name: valid_time_by_case_month[case_name][init_month]
        for case_name in E3SM_CASES
    }
    leads = VALID_LEADS_BY_MONTH[init_month][: RUN["seasonal_nlead"]]
    expected_cohorts = {
        case_name: {int(lead): expected_years_by_case_month[case_name][init_month][int(lead)] for lead in leads}
        for case_name in E3SM_CASES
    }
    first_expected = expected_cohorts[next(iter(expected_cohorts))]
    if any(cohort != first_expected for cohort in expected_cohorts.values()):
        raise ValueError(f"Cases have different configured target years for init={init_month}")
    land_skill.validate_reference_time_coverage(
        reference, first_expected, model_times[next(iter(model_times))],
        RUN["climatology_years"],
    )
    common_years = stats.common_valid_target_years_seasonal(
        model_fields, model_times, reference, leads, require_all_members=True
    )
    if common_years != first_expected:
        raise ValueError(
            f"Evaluation cohort is incomplete for init={init_month}: "
            f"expected={first_expected}, common={common_years}"
        )
    common_years_by_month[init_month] = common_years
    climy0, climy1 = RUN["climatology_years"]
    model_climatology_count_by_month[init_month] = {
        lead: sum(climy0 <= year <= climy1 for year in years)
        for lead, years in common_years.items()
    }
    print(init_month, {lead: (min(years), max(years), len(years)) for lead, years in common_years.items()})
    print("model climatology N by lead:", model_climatology_count_by_month[init_month])

## Compute and cache ACC metrics

In [ ]:
climy0, climy1 = RUN["climatology_years"]
trend_tag = "detrend" if RUN["detrend"] else "nodetrend"
reference_cache_token = safe_token(REFERENCE_PRODUCT)
skill_by_case_month = {}
skill_paths_by_case_month = {}

for case_name, case_cfg in E3SM_CASES.items():
    skill_by_case_month[case_name] = {}
    skill_paths_by_case_month[case_name] = {}
    outdir = leadtime_acc_dir(
        case_cfg["cache_tag"], "skill", "land", field, root=S2D_DIAG_ROOT
    )
    outdir.mkdir(parents=True, exist_ok=True)
    for init_month in RUN["init_months"]:
        cohorts = common_years_by_month[init_month]
        cohort_token = land_skill.land_cohort_token(cohorts)
        depth_part = f"_{DEPTH_TOKEN}" if DEPTH_TOKEN else ""
        filename = (
            f"{case_cfg['cache_tag']}{init_month:02d}_{field}{depth_part}_"
            f"{reference_cache_token}_skill_{cohort_token}_"
            f"clim_{climy0}_{climy1}_{safe_token(GRID_TAG)}_{trend_tag}.nc"
        )
        outfile = outdir / filename
        skill_paths_by_case_month[case_name][init_month] = outfile
        expected_attrs = land_skill.expected_land_skill_attrs(
            field=field, init_month=init_month,
            climatology_years=RUN["climatology_years"],
            initialization_years=RUN["initialization_years"],
            ensemble_members=RUN["ensemble_members"],
            target_years_by_lead=cohorts,
            reference_product=REFERENCE_PRODUCT,
            reference_data_identity=reference_data_identity,
            model_data_identity=model_data_identity_by_case_month[case_name][init_month],
            target_grid=GRID_TAG, detrend=RUN["detrend"],
            evaluation_protocol=EVALUATION_PROTOCOL,
            depth_range_m=SOIL_DEPTH_RANGE_M,
        )
        cache_ok, cache_reason = land_skill.land_skill_cache_status(
            outfile, expected_attrs, cohorts
        )
        if cache_ok and not RUN["force_compute"]:
            skill = load_netcdf(outfile)
            print(f"Loading compatible skill cache: {outfile}")
        else:
            if outfile.exists() and not RUN["force_compute"]:
                print(f"Recomputing incompatible cache {outfile}: {cache_reason}")
            skill = land_skill.compute_land_acc_skill(
                forecast_by_case_month[case_name][init_month],
                valid_time_by_case_month[case_name][init_month],
                reference, climy0, climy1,
                nleads=min(RUN["seasonal_nlead"], len(common_years_by_month[init_month])),
                detrend=RUN["detrend"],
                reference_is_anomaly=reference_is_anomaly,
                target_years_by_lead=common_years_by_month[init_month],
            ).compute()
            skill.attrs.update(expected_attrs)
            skill.attrs.update({
                "reference_product": REFERENCE_PRODUCT,
                "sample_alignment": "complete configured target years shared across E3SM cases and reference by lead",
                "evaluation_protocol": EVALUATION_PROTOCOL,
                "pointwise_significance": "effective-correlation p-value; no spatial multiple-testing correction",
            })
            atomic_to_netcdf(skill, outfile, **netcdf_write_options)
        land_skill.validate_land_skill_dataset(skill, expected_attrs, cohorts)
        skill_by_case_month[case_name][init_month] = skill
        print(outfile, dict(skill.sizes), "N=", skill.sample_count.values.tolist())

## Combined significance-masked ACC figure

This follows the multi-panel style of `1a_refactor_atm_leadtime_acc_skill_map.ipynb`: rows are valid seasonal leads, columns are E3SM cases grouped under the configured initialization-month headers, and only the outer axes carry latitude/longitude labels.

In [ ]:
import cartopy.crs as ccrs

from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import mov_utils as mov

spec = land_skill.get_land_variable_spec(field)

VARIABLE_DISPLAY_NAME = spec.plot_name
VARIABLE_TITLE_CONTEXT = (
    f"{VARIABLE_DISPLAY_NAME}, {REFERENCE_LEVEL_DEPTH}"
    if field == "H2OSOI" else VARIABLE_DISPLAY_NAME
)
VARIABLE_DIFFERENCE_CONTEXT = (
    f"{VARIABLE_DISPLAY_NAME}; {REFERENCE_LEVEL_DEPTH}"
    if field == "H2OSOI" else VARIABLE_DISPLAY_NAME
)
VARIABLE_CAPTION_NAME = (
    REFERENCE_CAPTION_NAME
    if field == "H2OSOI" else VARIABLE_DISPLAY_NAME.lower()
)

DETREND_DISPLAY_NAME = (
    "linear detrend" if RUN["detrend"] else "no detrend"
)
PLOT = FIGURE_SETTINGS["acc"]

case_display_names = {
    case_name: case_cfg.get("display_name", case_name)
    for case_name, case_cfg in E3SM_CASES.items()
}

column_specs = [
    (init_month, case_name)
    for init_month in RUN["init_months"]
    for case_name in E3SM_CASES
    if init_month in skill_by_case_month.get(case_name, {})
]
FULL_SEASONAL_LEADS = list(range(3, RUN["monthly_nlead"], 3))[
    : RUN["seasonal_nlead"]
]
plot_leads_by_month = {
    month: FULL_SEASONAL_LEADS for month in RUN["init_months"]
}
if not column_specs or not any(plot_leads_by_month.values()):
    raise RuntimeError("No common case/month/lead combinations are available for plotting")

corr_by_case_month = {}
for init_month, case_name in column_specs:
    skill = skill_by_case_month[case_name][init_month]
    corr = skill.corr.where(skill.valid_sample_count == skill.sample_count)
    if PLOT["significance_mask"]:
        corr = corr.where(skill.pval < PLOT["significance_level"])
    if PLOT["mask_negative_acc"]:
        corr = corr.where(corr >= 0)
    corr_by_case_month[(init_month, case_name)] = corr

nrows = max(map(len, plot_leads_by_month.values()))
ncols = len(column_specs)
projection = ccrs.PlateCarree()
fig = plt.figure(
    figsize=(PLOT["column_width"] * ncols, PLOT["row_height"] * nrows)
)
mappable = None

for row in range(nrows):
    for col, (init_month, case_name) in enumerate(column_specs):
        month_leads = plot_leads_by_month[init_month]
        subplot = row * ncols + col + 1
        lead = month_leads[row]
        skill = skill_by_case_month[case_name][init_month]
        title = case_display_names.get(case_name, case_name) if row == 0 else ""
        if lead in skill.L.values:
            ax, mappable = maps.map_pcolor_global_subplot(
                fig, corr_by_case_month[(init_month, case_name)].sel(L=lead),
                skill.lon, skill.lat,
                PLOT["color_interval"], PLOT["color_min"], PLOT["color_max"],
                title, nrows, ncols, subplot, projection,
                cmap=PLOT["cmap"], cutoff=PLOT["color_cutoff"],
                fontsize=PLOT["font_size"],
            )
        else:
            ax = add_missing_map_panel(
                fig, nrows=nrows, ncols=ncols, subplot=subplot, title=title,
                message=f"No {REFERENCE_DISPLAY_NAME}\nobservations",
                font_size=PLOT["font_size"],
            )
        style_global_map_axis(
            ax, row=row, column=col, nrows=nrows,
            label_size=PLOT["font_size"] * 0.75, style=MAP_STYLE,
        )
        if col == 0 or (col > 0 and column_specs[col - 1][0] != init_month):
            display_lead = int(lead) - 2
            add_lead_badge(
                ax, f"{display_lead:2d}: {seasonal_label(init_month, display_lead)}",
                font_size=PLOT["font_size"] * 0.75,
            )

skill_figure_title = (
    f"ACC ({VARIABLE_TITLE_CONTEXT}, ref: {REFERENCE_DISPLAY_NAME}; "
    f"{DETREND_DISPLAY_NAME})"
)
fig.suptitle(
    skill_figure_title,
    fontsize=PLOT["font_size"] * 1.2, fontweight="bold", y=0.995,
)
layout_top = 0.90 if nrows <= 4 else 0.96
section_header_y = 0.925 if nrows <= 4 else 0.962
divider_top = 0.915 if nrows <= 4 else 0.95
fig.tight_layout(rect=[0.0, 0.075, 1.0, layout_top])
fig.subplots_adjust(bottom=0.09, hspace=0.06, wspace=0.025)

# Add a small gap and divider between initialization-month groups.
second_month = RUN["init_months"][1] if len(RUN["init_months"]) > 1 else None
if second_month is not None:
    second_cols = [i for i, (month, _) in enumerate(column_specs) if month == second_month]
    for row in range(nrows):
        for col in second_cols:
            ax = fig.axes[row * ncols + col]
            pos = ax.get_position()
            ax.set_position([pos.x0 + PLOT["section_gap"], pos.y0, pos.width, pos.height])

for init_month in RUN["init_months"]:
    cols = [i for i, (month, _) in enumerate(column_specs) if month == init_month]
    if cols:
        left = min(fig.axes[col].get_position().x0 for col in cols)
        right = max(fig.axes[col].get_position().x1 for col in cols)
        fig.text(
            (left + right) / 2, section_header_y, f"{init_month_names.get(init_month, init_month)} initialization",
            ha="center", va="bottom", fontsize=PLOT["font_size"], fontweight="bold",
        )

if second_month is not None:
    first_cols = [i for i, (month, _) in enumerate(column_specs) if month != second_month]
    second_cols = [i for i, (month, _) in enumerate(column_specs) if month == second_month]
    divider_x = (
        max(fig.axes[col].get_position().x1 for col in first_cols)
        + min(fig.axes[col].get_position().x0 for col in second_cols)
    ) / 2
    fig.add_artist(plt.Line2D(
        [divider_x, divider_x], [0.08, divider_top], transform=fig.transFigure,
        color="0.25", linewidth=1.0,
    ))

colorbar_ax = fig.add_axes([0.28, 0.025, 0.44, 0.015])
colorbar = fig.colorbar(mappable, cax=colorbar_ax, orientation="horizontal")
colorbar.set_label("ACC", fontsize=PLOT["font_size"], fontweight="bold")
colorbar.ax.tick_params(labelsize=PLOT["font_size"] * 0.8)

mask_tag = (
    f"sigmask_p{int(PLOT['significance_level'] * 100):02d}"
    if PLOT["significance_mask"] else "no_sigmask"
)
figure_cohort_token = "_".join(
    f"init{month:02d}_{land_skill.land_cohort_token(common_years_by_month[month])}"
    for month in RUN["init_months"]
)
figure_name = figure_filename(
    field, DEPTH_TOKEN, REFERENCE_PRODUCT, figure_cohort_token,
    f"clim_{climy0}_{climy1}", trend_tag, "multi_e3sm_acc", mask_tag,
)
figure_path = FIGURE_OUTDIR / figure_name
mov.save_figure(
    fig, figure_path, mode="", metric="leadtime_acc",
    title=skill_figure_title,
    caption=(f"ACC rows by seasonal lead using complete configured cohorts for "
             f"initializations {RUN['initialization_years'][0]}–{RUN['initialization_years'][1]} "
             f"and climatology {climy0}–{climy1}; "
             f"{'linear detrending' if RUN['detrend'] else 'no detrending'}; "
             "pointwise effective-correlation p < 0.1 without spatial "
             "multiple-testing correction."),
    dpi=PLOT["dpi"],
)
print("Saved figure:", figure_path)
plt.show()

## FOSIRL minus Reanalysis ACC difference

This descriptive comparison uses the same lead-by-initialization layout. Positive values indicate higher ACC for E3SMv3-FOSIRL; no difference-significance stippling is shown because the cached skill files do not contain paired resampling statistics for the ACC difference.

In [ ]:
DIFF_PLOT = FIGURE_SETTINGS["difference"]
comparison_cases = ("E3SM-FOSIRL", "E3SM-Reanalysis")
missing_cases = [case for case in comparison_cases if case not in skill_by_case_month]
if missing_cases:
    raise RuntimeError(f"Missing comparison cases: {missing_cases}")

difference_months = [
    month for month in RUN["init_months"]
    if all(month in skill_by_case_month[case] for case in comparison_cases)
]
difference_leads_by_month = {
    month: FULL_SEASONAL_LEADS for month in difference_months
}
if not difference_months or not any(difference_leads_by_month.values()):
    raise RuntimeError("No common month/lead combinations are available for the ACC difference")

acc_difference = {}
for init_month in difference_months:
    fosirl_skill = skill_by_case_month[comparison_cases[0]][init_month]
    reanalysis_skill = skill_by_case_month[comparison_cases[1]][init_month]
    fosirl = fosirl_skill.corr.where(
        fosirl_skill.valid_sample_count == fosirl_skill.sample_count
    )
    reanalysis = reanalysis_skill.corr.where(
        reanalysis_skill.valid_sample_count == reanalysis_skill.sample_count
    )
    acc_difference[init_month] = fosirl - reanalysis

nrows = max(map(len, difference_leads_by_month.values()))
ncols = len(difference_months)
fig = plt.figure(
    figsize=(DIFF_PLOT["column_width"] * ncols, DIFF_PLOT["row_height"] * nrows)
)
mappable = None
for row in range(nrows):
    for col, init_month in enumerate(difference_months):
        month_leads = difference_leads_by_month[init_month]
        subplot = row * ncols + col + 1
        lead = month_leads[row]
        skill = skill_by_case_month[comparison_cases[0]][init_month]
        title = (
            f"{init_month_names.get(init_month, init_month)} initialization"
            if row == 0 else ""
        )
        if lead in acc_difference[init_month].L.values:
            ax, mappable = maps.map_pcolor_global_subplot(
                fig, acc_difference[init_month].sel(L=lead), skill.lon, skill.lat,
                DIFF_PLOT["color_interval"], DIFF_PLOT["color_min"], DIFF_PLOT["color_max"],
                title, nrows, ncols, subplot, projection,
                cmap=DIFF_PLOT["cmap"], cutoff=DIFF_PLOT["color_cutoff"],
                fontsize=DIFF_PLOT["font_size"],
            )
        else:
            ax = add_missing_map_panel(
                fig, nrows=nrows, ncols=ncols, subplot=subplot, title=title,
                message=f"No {REFERENCE_DISPLAY_NAME}\nobservations",
                font_size=DIFF_PLOT["font_size"],
            )
        style_global_map_axis(
            ax, row=row, column=col, nrows=nrows,
            label_size=DIFF_PLOT["font_size"] * 0.75, style=MAP_STYLE,
        )
        if col == 0:
            display_lead = int(lead) - 2
            add_lead_badge(
                ax, f"{display_lead:2d}: {seasonal_label(init_month, display_lead)}",
                font_size=DIFF_PLOT["font_size"] * 0.75,
            )

difference_title = DIFF_PLOT["title_template"].format(
    variable=VARIABLE_DIFFERENCE_CONTEXT,
    reference=REFERENCE_DISPLAY_NAME,
)
difference_caption = (
    f"FOSIRL minus Reanalysis ACC for {VARIABLE_CAPTION_NAME} relative "
    f"to {REFERENCE_DISPLAY_NAME}, by seasonal lead and initialization month; "
    f"initializations {RUN['initialization_years'][0]}–{RUN['initialization_years'][1]}, "
    f"climatology {climy0}–{climy1}, {DETREND_DISPLAY_NAME}."
)
figure_height_points = fig.get_figheight() * 72.0
difference_subtitle = DIFF_PLOT["subtitle_template"].format(
    left=case_display_names[comparison_cases[0]],
    right=case_display_names[comparison_cases[1]],
)
title_artist = fig.suptitle(
    difference_title,
    fontsize=DIFF_PLOT["font_size"] * DIFF_PLOT["title_font_scale"],
    fontweight="bold", va="top",
    y=1.0 - DIFF_PLOT["header_top_pad_points"] / figure_height_points,
)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
title_box = title_artist.get_window_extent(renderer).transformed(
    fig.transFigure.inverted()
)
subtitle_artist = fig.text(
    0.5,
    title_box.y0 - DIFF_PLOT["title_subtitle_gap_points"] / figure_height_points,
    difference_subtitle,
    ha="center", va="top",
    fontsize=DIFF_PLOT["font_size"] * DIFF_PLOT["subtitle_font_scale"],
)
fig.canvas.draw()
subtitle_box = subtitle_artist.get_window_extent(renderer).transformed(
    fig.transFigure.inverted()
)
layout_top = (
    subtitle_box.y0
    - DIFF_PLOT["header_panel_gap_points"] / figure_height_points
)
layout_left, layout_bottom, layout_right = DIFF_PLOT["layout_edges"]
fig.tight_layout(
    rect=(layout_left, layout_bottom, layout_right, layout_top),
    pad=DIFF_PLOT["layout_pad"],
    h_pad=DIFF_PLOT["layout_h_pad"],
    w_pad=DIFF_PLOT["layout_w_pad"],
)
fig.subplots_adjust(
    bottom=DIFF_PLOT["subplot_bottom"],
    hspace=DIFF_PLOT["subplot_hspace"],
    wspace=DIFF_PLOT["subplot_wspace"],
)
colorbar_ax = fig.add_axes(DIFF_PLOT["colorbar_rect"])
colorbar = fig.colorbar(mappable, cax=colorbar_ax, orientation="horizontal")
colorbar.set_label(
    DIFF_PLOT["colorbar_label"],
    fontsize=DIFF_PLOT["font_size"], fontweight="bold",
)
colorbar.ax.tick_params(
    labelsize=(
        DIFF_PLOT["font_size"] * DIFF_PLOT["colorbar_tick_font_scale"]
    )
)

difference_name = figure_filename(
    field, DEPTH_TOKEN, REFERENCE_PRODUCT, figure_cohort_token,
    f"clim_{climy0}_{climy1}", trend_tag,
    comparison_cases[0], "minus", comparison_cases[1], "acc",
)
difference_path = FIGURE_OUTDIR / difference_name
mov.save_figure(
    fig, difference_path, mode="", metric="leadtime_acc_difference",
    title=difference_title,
    caption=difference_caption,
    dpi=DIFF_PLOT["dpi"],
)
print("Saved difference figure:", difference_path)
plt.show()


## Cleanup

Prepared inputs, skills, and figures remain on disk. Run this after completion or an
interrupted calculation to close staged datasets, the Dask client, and its cluster.


In [ ]:
close_notebook_resources(globals())
print("Closed land-workflow datasets and Dask resources.")
